# DeBERTa-v3-small Fine-Tuning â€” GECS Classification (Colab)

**Goal:** Fine-tune `microsoft/deberta-v3-small` on Morningstar company descriptions and beat the TF-IDF + LinearSVM baseline of **86.82%** Macro F1.

This notebook mirrors `llm_finetuning/scripts/train_local.py` but is tuned for Colab T4/A100 GPUs. With 16+ GB of VRAM we can use `batch_size=16` directly (no gradient accumulation) and finish in ~60â€“90 minutes on a free T4.

**Setup:** `Runtime > Change runtime type > T4 GPU` (free) or A100 (Pro+).

## How to use
1. Upload `task1_train.csv`, `task1_test.csv`, and the two label-map JSONs to a Drive folder called `capstone_llm`
2. Run all cells top to bottom
3. Best checkpoint and results land back in `capstone_llm/results/` on your Drive

## Why this notebook differs from the original
The first version used HuggingFace `Trainer`. On our setup that produced gradient explosions (Trainer + Accelerate + DeBERTa-v3 had a known incompatibility â€” see Issue 6 in the docs). This notebook uses a raw PyTorch loop, which is what the local script also uses. Same behavior, no surprises.

## 1. Install dependencies

Pin `transformers` to `4.44.2`. Newer versions broke DeBERTa-v3 fine-tuning (Issue 8). `sentencepiece` is required by the DeBERTa tokenizer.

In [ ]:
!pip install -q transformers==4.44.2 sentencepiece==0.2.0

## 2. Mount Drive and verify files

Upload these four files to `MyDrive/capstone_llm/`:
```
task1_train.csv
task1_test.csv
task1_code_to_idx.json
task1_idx_to_code.json
```
(For Task 2, also upload the corresponding `task2_*` files.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR    = '/content/drive/MyDrive/capstone_llm'
RESULTS_DIR = '/content/drive/MyDrive/capstone_llm/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

for f in ['task1_train.csv', 'task1_test.csv', 'task1_code_to_idx.json', 'task1_idx_to_code.json']:
    path = os.path.join(DATA_DIR, f)
    print(f"  {'OK ' if os.path.exists(path) else 'MISS'}  {f}")

## 3. Configuration

All knobs live here. The defaults match what we know works.

| Knob | Value | Why |
|---|---|---|
| `MAX_LEN` | 128 | Task 1 text averages 89 words â€” fits comfortably |
| `BATCH_SIZE` | 16 | T4 has 16 GB VRAM â€” no need for gradient accumulation |
| `EPOCHS` | 6 | Standard for transformer fine-tuning |
| `LR` | 2e-5 | Canonical DeBERTa starting point |
| `WARMUP_PCT` | 0.1 | 10% warmup prevents the random head from corrupting pretrained weights |
| Precision | FP32 | DeBERTa-v3 is unstable in BF16/FP16 |

In [ ]:
MODEL_NAME   = 'microsoft/deberta-v3-small'
MAX_LEN      = 128
BATCH_SIZE   = 16
EVAL_BS      = 32
EPOCHS       = 10
LR           = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_PCT   = 0.1
SEED         = 42
LOG_EVERY    = 100

TASK        = 'task1'
BASELINE_F1 = 0.8682

print(f'Model : {MODEL_NAME}')
print(f'Task  : {TASK}')
print(f'Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}')

## 4. Load data and tokenizer

In [ ]:
import json
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report

torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU â€” switch runtime to T4 before continuing.')

train_df = pd.read_csv(os.path.join(DATA_DIR, f'{TASK}_train.csv'),
                       dtype={'text': str, 'label_idx': int})
test_df  = pd.read_csv(os.path.join(DATA_DIR, f'{TASK}_test.csv'),
                       dtype={'text': str, 'label_idx': int})
with open(os.path.join(DATA_DIR, f'{TASK}_idx_to_code.json')) as f:
    idx_to_code = {int(k): v for k, v in json.load(f).items()}

NUM_LABELS = train_df['label_idx'].nunique()
print(f'\nTrain: {len(train_df)} rows  |  Test: {len(test_df)} rows  |  Classes: {NUM_LABELS}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## 5. Tokenize and build DataLoaders

Tokenize all data upfront â€” at this scale (40kâ€“60k rows) it's faster than tokenizing per-batch. Each sample becomes a fixed 128-token sequence.

In [ ]:
class GECSDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()
                if k in ('input_ids', 'attention_mask')}
        item['labels'] = self.labels[idx]
        return item

print('Tokenizing train ...')
train_ds = GECSDataset(train_df['text'].tolist(), train_df['label_idx'].tolist(), tokenizer)
print('Tokenizing test ...')
test_ds  = GECSDataset(test_df['text'].tolist(),  test_df['label_idx'].tolist(),  tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=EVAL_BS,    shuffle=False, num_workers=2)

print(f'Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}')

## 6. Load model + optimizer

AdamW with two parameter groups â€” weights get weight decay, biases and LayerNorm parameters do not. This is the standard transformer fine-tuning setup.

In [ ]:
ckpt_dir = os.path.join(RESULTS_DIR, f'{TASK}_best_model')

# Load from saved checkpoint (epoch 5, 60.67% F1) — skip retraining epochs 1-5
model = AutoModelForSequenceClassification.from_pretrained(
    ckpt_dir, num_labels=NUM_LABELS
).to(device)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Loaded checkpoint from {ckpt_dir}')
print(f'Parameters: {total_params:.1f}M')

no_decay = ['bias', 'LayerNorm.weight']
params = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': WEIGHT_DECAY},
    {'params': [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)],
     'weight_decay': 0.0},
]
optimizer = torch.optim.AdamW(params, lr=LR)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = 0  # no warmup — resuming from checkpoint
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f'Total steps: {total_steps}  |  Warmup: {warmup_steps}')

## 7. Training loop

Raw PyTorch â€” same logic as the local script. After each epoch, evaluate on the test set, save the best checkpoint, and clear cache between epochs.

In [ ]:
import time
import gc

def train_epoch(epoch):
    model.train()
    total_loss, window_loss = 0.0, 0.0
    start = time.time()

    for step, batch in enumerate(train_loader):
        batch  = {k: v.to(device) for k, v in batch.items()}
        output = model(**batch)
        loss   = output.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        total_loss  += loss.item()
        window_loss += loss.item()

        if (step + 1) % LOG_EVERY == 0 or (step + 1) == len(train_loader):
            avg = window_loss / LOG_EVERY if (step + 1) % LOG_EVERY == 0 \
                  else window_loss / ((step + 1) % LOG_EVERY or LOG_EVERY)
            elapsed = time.time() - start
            pct = (step + 1) / len(train_loader) * 100
            print(f'  Epoch {epoch} | step {step+1}/{len(train_loader)} | '
                  f'loss {avg:.4f} | lr {scheduler.get_last_lr()[0]:.2e} | '
                  f'{pct:.1f}% | {elapsed/60:.1f}min', flush=True)
            window_loss = 0.0

    return total_loss / len(train_loader)

def evaluate():
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch  = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop('labels')
            preds  = model(**batch).logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return f1, all_preds, all_labels

best_f1, best_epoch = 0.0, 0
ckpt_dir = os.path.join(RESULTS_DIR, f'{TASK}_best_model')

for epoch in range(1, EPOCHS + 1):
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\n--- Epoch {epoch}/{EPOCHS} ---', flush=True)
    avg_loss = train_epoch(epoch)

    torch.cuda.empty_cache()
    f1, _, _ = evaluate()
    torch.cuda.empty_cache()
    print(f'  Epoch {epoch} | avg loss {avg_loss:.4f} | Macro F1: {f1*100:.2f}%', flush=True)

    if f1 > best_f1:
        best_f1, best_epoch = f1, epoch
        model.save_pretrained(ckpt_dir)
        tokenizer.save_pretrained(ckpt_dir)
        print(f'  New best â€” saved to {ckpt_dir}', flush=True)

print(f'\nBest epoch: {best_epoch}  |  Best Macro F1: {best_f1*100:.2f}%')

## 7b. Resume training from checkpoint

**Run this cell instead of Cell 7 if your training was interrupted.**

Loads the best saved checkpoint from Drive and continues for `RESUME_EPOCHS` more epochs at a lower learning rate. No warmup — the model is already adapted.

Before running:
1. Run cells 1–5 (pip, drive mount, config, data load, tokenize/DataLoaders)
2. Update `BEST_F1_SO_FAR` below to match the best F1 you saw before interruption
3. Run **this** cell, then skip Cell 7 and continue from Cell 8 onward

In [ ]:
import time, gc

RESUME_EPOCHS   = 3       # extra epochs to run from the checkpoint
RESUME_LR       = 1e-5    # lower LR — model is already well-adapted
BEST_F1_SO_FAR  = 0.6067  # update this to the best F1 seen before you paused

ckpt_dir = os.path.join(RESULTS_DIR, f'{TASK}_best_model')
print(f'Loading checkpoint from {ckpt_dir} ...')
model = AutoModelForSequenceClassification.from_pretrained(
    ckpt_dir, num_labels=NUM_LABELS
).to(device)
print('Checkpoint loaded.')

no_decay = ['bias', 'LayerNorm.weight']
params = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': WEIGHT_DECAY},
    {'params': [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)],
     'weight_decay': 0.0},
]
optimizer = torch.optim.AdamW(params, lr=RESUME_LR)
total_steps = len(train_loader) * RESUME_EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer, 0, total_steps)  # no warmup

print(f'Resuming: {RESUME_EPOCHS} epochs | LR {RESUME_LR:.0e} | total steps {total_steps}')

best_f1    = BEST_F1_SO_FAR
best_epoch = 'pre-resume'

for epoch in range(1, RESUME_EPOCHS + 1):
    gc.collect()
    torch.cuda.empty_cache()
    label = f'R{epoch}'
    print(f'
--- Resume Epoch {epoch}/{RESUME_EPOCHS} ---', flush=True)
    avg_loss = train_epoch(label)

    torch.cuda.empty_cache()
    f1, _, _ = evaluate()
    torch.cuda.empty_cache()
    print(f'  Resume Epoch {epoch} | avg loss {avg_loss:.4f} | Macro F1: {f1*100:.2f}%', flush=True)

    if f1 > best_f1:
        best_f1, best_epoch = f1, label
        model.save_pretrained(ckpt_dir)
        tokenizer.save_pretrained(ckpt_dir)
        print(f'  New best — saved to {ckpt_dir}', flush=True)

print(f'
Best: epoch {best_epoch}  |  Macro F1: {best_f1*100:.2f}%')
print('
Continue from Cell 8 (Final Evaluation) to get the full results.')

## 8. Final evaluation on the best checkpoint

Reload the best epoch and produce the official Macro F1 alongside the per-class breakdown.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    ckpt_dir, num_labels=NUM_LABELS
).to(device)

final_f1, y_pred, y_true = evaluate()
delta = final_f1 - BASELINE_F1

print('=' * 50)
print(f'  Baseline (TF-IDF + SVM) : {BASELINE_F1*100:.2f}%')
print(f'  DeBERTa-v3-small        : {final_f1*100:.2f}%')
print(f'  Delta                   : {delta*100:+.2f}%')
print('  Baseline beaten.' if delta > 0 else '  Did not beat baseline.')
print('=' * 50)

## 9. Save results JSON

In [ ]:
target_names = [str(idx_to_code[i]) for i in range(NUM_LABELS)]
report = classification_report(y_true, y_pred, target_names=target_names,
                               output_dict=True, zero_division=0)

summary = {
    'task': TASK,
    'model': MODEL_NAME,
    'max_len': MAX_LEN,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LR,
    'num_labels': NUM_LABELS,
    'baseline_macro_f1': BASELINE_F1,
    'deberta_macro_f1': round(final_f1, 6),
    'delta': round(delta, 6),
    'best_epoch': best_epoch,
    'per_class': report,
}

out_path = os.path.join(RESULTS_DIR, f'{TASK}_results.json')
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved to {out_path}')

## 10. Visualise the 25 weakest classes

These are the classes the model struggles with â€” useful for identifying where more training data or label cleanup would help most.

In [ ]:
import matplotlib.pyplot as plt

report_df = pd.DataFrame(report).T
report_df = report_df.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
report_df = report_df[['f1-score', 'support']].astype(float)
bottom25 = report_df.sort_values('f1-score').head(25)

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(bottom25.index.astype(str), bottom25['f1-score'], color='steelblue')
ax.axvline(x=BASELINE_F1, color='red', linestyle='--', linewidth=1.5,
           label=f'Baseline {BASELINE_F1*100:.1f}%')
ax.axvline(x=final_f1,  color='green', linestyle='--', linewidth=1.5,
           label=f'DeBERTa {final_f1*100:.1f}%')
ax.set_xlabel('F1 Score')
ax.set_title(f'25 Weakest Classes â€” {TASK} DeBERTa-v3-small')
ax.legend()
plt.tight_layout()

fig_path = os.path.join(RESULTS_DIR, f'{TASK}_weak_classes.png')
plt.savefig(fig_path, dpi=150)
plt.show()
print(f'Plot saved to {fig_path}')

## 11. To train Task 2

Go back to **Cell 3 (Configuration)** and change:
```python
TASK        = 'task2'
BASELINE_F1 = 0.0   # no baseline yet for task2
```
Make sure `task2_train.csv`, `task2_test.csv`, and the two label-map JSONs are uploaded to your Drive folder. Then re-run from Cell 3 onward.

Task 2 has 407 classes (vs 145 for Task 1) and shorter text â€” it's a harder problem. Expect Macro F1 to be lower in absolute terms, but the relative improvement over an eventual TF-IDF baseline will be larger.